# 16 — Probe-size and α-rule audit

This notebook is a **descriptive audit of already frozen artifacts** for reviewer response.

It does not train a model, embed new images, score TEST with a detector, or retune the allocator.


In [ ]:
from pathlib import Path
import json, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
manifest=pd.read_csv(config['dataset']['prepared_manifest'])
test=manifest[manifest.split=='test'].reset_index(drop=True)
test['source_id']=test.source_id.astype(str)
assert len(test)==2000 and test.source_id.is_unique

out06=Path('/workspace/results/frozen_test_final')
protocol06=json.loads((out06/'test_protocol.json').read_text())
context_tag=protocol06['context_tag']
context_dir=out06/'contexts'/context_tag
if not context_dir.exists():
    raise RuntimeError(f'Frozen context cache missing: {context_dir}')

OUT=Path('/workspace/results/reviewer2_audit_v16')
OUT.mkdir(parents=True,exist_ok=True)

print('TEST sources:',len(test))
print('Context tag:',context_tag)
print('Context directory:',context_dir)


In [ ]:
rows=[]
missing=[]
for k,sid in enumerate(test.source_id.tolist(),1):
    p=context_dir/f'{sid}.joblib'
    if not p.exists():
        missing.append(sid)
        continue
    ctx=joblib.load(p)
    if ctx.get('context_tag')!=context_tag:
        raise RuntimeError(f'Stale context tag for {sid}')
    br=ctx['block_rows']
    for r in br:
        rows.append({
            'source_id':sid,
            'block_id':int(r['block_id']),
            'probe_bits':int(r.get('probe_bits',-1)),
            'detectability_risk':float(r['detectability_risk']),
            'predictability':float(r['predictability']),
        })
    if k%250==0:
        print('contexts',k,'/',len(test))

if missing:
    raise RuntimeError(f'Missing {len(missing)} frozen contexts; first few: {missing[:10]}')

df=pd.DataFrame(rows)
if len(df)!=32000:
    raise RuntimeError(f'Expected 32,000 block contexts, got {len(df)}')
if (df.probe_bits<0).any():
    raise RuntimeError('probe_bits missing from frozen block_rows')

hist=(df.groupby('probe_bits').size().rename('count').reset_index())
hist['fraction']=hist['count']/len(df)
hist.to_csv(OUT/'probe_bits_histogram.csv',index=False)

positive=df[df.probe_bits>0].copy()
q=positive.probe_bits.quantile([0.01,0.05,0.25,0.5,0.75,0.95,0.99])

summary={
  'analysis_status':'DESCRIPTIVE_FROZEN_TEST_CONTEXT_AUDIT',
  'test_sources':int(test.source_id.nunique()),
  'blocks_total':int(len(df)),
  'blocks_per_source':int(len(df)//test.source_id.nunique()),
  'probe_zero_count':int((df.probe_bits==0).sum()),
  'probe_zero_fraction':float((df.probe_bits==0).mean()),
  'positive_probe_blocks':int(len(positive)),
  'positive_probe_min':int(positive.probe_bits.min()),
  'positive_probe_mean':float(positive.probe_bits.mean()),
  'positive_probe_median':float(positive.probe_bits.median()),
  'positive_probe_max':int(positive.probe_bits.max()),
  'positive_probe_p01':float(q.loc[0.01]),
  'positive_probe_p05':float(q.loc[0.05]),
  'positive_probe_p25':float(q.loc[0.25]),
  'positive_probe_p75':float(q.loc[0.75]),
  'positive_probe_p95':float(q.loc[0.95]),
  'positive_probe_p99':float(q.loc[0.99]),
  'probe_1bit_count':int((positive.probe_bits==1).sum()),
  'probe_1bit_fraction_positive':float((positive.probe_bits==1).mean()),
  'probe_le4_fraction_positive':float((positive.probe_bits<=4).mean()),
  'probe_le8_fraction_positive':float((positive.probe_bits<=8).mean()),
  'probe_le16_fraction_positive':float((positive.probe_bits<=16).mean()),
  'probe_ge32_fraction_positive':float((positive.probe_bits>=32).mean()),
  'probe_128_fraction_positive':float((positive.probe_bits==128).mean()),
  'context_tag':context_tag,
}
(OUT/'probe_bits_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))

bins=[0,4,8,16,32,64,127,128]
labels=['1-4','5-8','9-16','17-32','33-64','65-127','128']
positive['probe_bin']=pd.cut(
    positive.probe_bits,
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)
risk_bins=(positive.groupby('probe_bin',observed=False)
           .agg(n=('detectability_risk','size'),
                probe_mean=('probe_bits','mean'),
                risk_mean=('detectability_risk','mean'),
                risk_std=('detectability_risk','std'),
                risk_abs_median=('detectability_risk',lambda x: float(np.median(np.abs(x)))))
           .reset_index())
risk_bins['fraction_positive']=risk_bins['n']/len(positive)
risk_bins.to_csv(OUT/'probe_bits_risk_bins.csv',index=False)
display(risk_bins)


In [ ]:
fig,ax=plt.subplots(figsize=(7.0,4.4))
ax.hist(positive.probe_bits,bins=np.arange(0.5,129.5,4),edgecolor='black',linewidth=.4)
ax.set_xlabel('Probe bits per 64×64 block')
ax.set_ylabel('Number of frozen TEST blocks')
ax.set_title('Distribution of local reversible probe size')
fig.tight_layout()
fig.savefig(OUT/'probe_bits_histogram.png',dpi=300)
plt.show()


In [ ]:
alpha_path=Path('/workspace/results/srm_teacher_risk/alpha_validation_summary.csv')
s=pd.read_csv(alpha_path).sort_values('alpha').copy()
r0=s.loc[np.isclose(s.alpha,0.0)].iloc[0]
r1=s.loc[np.isclose(s.alpha,1.0)].iloc[0]
max_teacher_gain=float(r1.teacher_delta_median-r0.teacher_delta_median)
if max_teacher_gain<=0:
    raise RuntimeError('Unexpected validation teacher-gain direction.')

s['teacher_gain_retained']=(float(r1.teacher_delta_median)-s.teacher_delta_median)/max_teacher_gain
s.to_csv(OUT/'alpha_validation_with_gain_retained.csv',index=False)

rows=[]
for tau in [0.80,0.90,0.95]:
    eligible=s[
        (s.alpha>0) & (s.alpha<1) &
        (s.teacher_gain_retained>=tau) &
        (s.psnr_mean>float(r0.psnr_mean))
    ].sort_values('alpha')
    if len(eligible):
        rr=eligible.iloc[0]
        rows.append({
          'retention_threshold':tau,
          'selected_alpha':float(rr.alpha),
          'teacher_gain_retained':float(rr.teacher_gain_retained),
          'psnr_mean':float(rr.psnr_mean),
          'selection_exists':True,
        })
    else:
        rows.append({
          'retention_threshold':tau,
          'selected_alpha':np.nan,
          'teacher_gain_retained':np.nan,
          'psnr_mean':np.nan,
          'selection_exists':False,
        })

sens=pd.DataFrame(rows)
sens.to_csv(OUT/'alpha_threshold_sensitivity.csv',index=False)
display(s[['alpha','teacher_delta_median','psnr_mean','teacher_gain_retained']])
display(sens)


In [ ]:
complete={
  'status':'COMPLETE',
  'analysis_status':'REVIEWER2_DIAGNOSTIC_AUDIT_V16',
  'new_training':False,
  'new_embedding':False,
  'new_test_detector_scoring':False,
  'allocator_retuned':False,
  'frozen_alpha_unchanged':True,
  'probe_context_blocks':int(len(df)),
  'alpha_thresholds_checked':[0.80,0.90,0.95],
}
(OUT/'reviewer2_audit_v16_complete.json').write_text(json.dumps(complete,indent=2),encoding='utf-8')
print(json.dumps(complete,indent=2))
print('PATCH 16 COMPLETE')
print('Send:')
for name in [
  'probe_bits_summary.json',
  'probe_bits_histogram.csv',
  'probe_bits_risk_bins.csv',
  'alpha_threshold_sensitivity.csv',
  'alpha_validation_with_gain_retained.csv',
  'reviewer2_audit_v16_complete.json',
]:
    print(' -',OUT/name)
